In [1]:
# For tips on running notebooks in Google Colab, see
# https://docs.pytorch.org/tutorials/beginner/colab
%matplotlib inline

A Gentle Introduction to `torch.autograd`
=========================================

`torch.autograd` is PyTorch's automatic differentiation engine that
powers neural network training. In this section, you will get a
conceptual understanding of how autograd helps a neural network train.

Background
----------

Neural networks (NNs) are a collection of nested functions that are
executed on some input data. These functions are defined by *parameters*
(consisting of weights and biases), which in PyTorch are stored in
tensors.

Training a NN happens in two steps:

**Forward Propagation**: In forward prop, the NN makes its best guess
about the correct output. It runs the input data through each of its
functions to make this guess.

**Backward Propagation**: In backprop, the NN adjusts its parameters
proportionate to the error in its guess. It does this by traversing
backwards from the output, collecting the derivatives of the error with
respect to the parameters of the functions (*gradients*), and optimizing
the parameters using gradient descent. For a more detailed walkthrough
of backprop, check out this [video from
3Blue1Brown](https://www.youtube.com/watch?v=tIeHLnjs5U8).

Usage in PyTorch
----------------

Let\'s take a look at a single training step. For this example, we load
a pretrained resnet18 model from `torchvision`. We create a random data
tensor to represent a single image with 3 channels, and height & width
of 64, and its corresponding `label` initialized to some random values.
Label in pretrained models has shape (1,1000).

<div style="background-color: #54c7ec; color: #fff; font-weight: 700; padding-left: 10px; padding-top: 5px; padding-bottom: 5px"><strong>NOTE:</strong></div>

<div style="background-color: #f3f4f7; padding-left: 10px; padding-top: 10px; padding-bottom: 10px; padding-right: 10px">

<p>This tutorial works only on the CPU and will not work on GPU devices (even if tensors are moved to CUDA).</p>

</div>



In [2]:
import torch
from torchvision.models import resnet18, ResNet18_Weights
model = resnet18(weights=ResNet18_Weights.DEFAULT)
data = torch.rand(1, 3, 64, 64)
print(data)
labels = torch.rand(1, 1000)
print(labels)

tensor([[[[0.2566, 0.2256, 0.4544,  ..., 0.1911, 0.6730, 0.8490],
          [0.7591, 0.4457, 0.9185,  ..., 0.7490, 0.0596, 0.5979],
          [0.2055, 0.1817, 0.1056,  ..., 0.7757, 0.7315, 0.9946],
          ...,
          [0.4646, 0.0870, 0.6871,  ..., 0.3691, 0.5227, 0.5507],
          [0.0375, 0.6313, 0.8707,  ..., 0.4904, 0.6144, 0.4317],
          [0.7017, 0.9620, 0.3062,  ..., 0.7146, 0.1627, 0.2882]],

         [[0.8821, 0.3965, 0.5261,  ..., 0.5055, 0.6531, 0.4353],
          [0.7282, 0.2877, 0.1676,  ..., 0.8818, 0.4926, 0.2652],
          [0.4385, 0.0252, 0.6291,  ..., 0.7303, 0.9776, 0.7468],
          ...,
          [0.6625, 0.1702, 0.8571,  ..., 0.3571, 0.3853, 0.3585],
          [0.9935, 0.6204, 0.1321,  ..., 0.0024, 0.7330, 0.5207],
          [0.6387, 0.1058, 0.6958,  ..., 0.1794, 0.1652, 0.5496]],

         [[0.3662, 0.7075, 0.8169,  ..., 0.2044, 0.6494, 0.5714],
          [0.7188, 0.5216, 0.0015,  ..., 0.5814, 0.7989, 0.4734],
          [0.2891, 0.0041, 0.3367,  ..., 0

Next, we run the input data through the model through each of its layers
to make a prediction. This is the **forward pass**.


In [3]:
prediction = model(data) # forward pass
print(prediction.dtype)
print(prediction.shape)
print(prediction)

torch.float32
torch.Size([1, 1000])
tensor([[-4.7917e-01, -2.0062e-01, -4.6313e-01, -1.4945e+00, -5.2620e-01,
         -1.3350e-03, -5.4742e-01,  7.3502e-01,  6.7789e-01, -9.4675e-01,
         -8.9989e-01, -9.5097e-01, -2.7894e-02, -7.5837e-01, -1.1176e+00,
         -3.6472e-01, -5.9014e-01, -2.4599e-01, -7.1233e-01, -3.4726e-01,
         -1.7769e+00, -5.5747e-01, -1.8071e+00,  5.5983e-02, -8.7472e-01,
         -1.4979e+00, -6.9193e-01, -1.0699e+00, -8.8502e-01, -2.1582e-01,
         -7.5973e-01, -8.4824e-01, -3.5086e-01, -4.8424e-01, -4.2906e-01,
         -6.8003e-01,  4.5979e-01, -8.2419e-01, -2.6627e-01, -7.5971e-03,
         -4.8071e-01, -6.6873e-01, -8.4190e-01, -9.3939e-02, -6.2981e-01,
         -5.0508e-01, -4.9951e-01, -5.0258e-01, -1.4851e+00, -1.2398e+00,
         -7.5446e-01,  3.2845e-01, -7.1693e-02, -6.2017e-01,  5.7746e-02,
         -1.0581e+00, -3.0200e-01, -1.3421e+00, -3.9215e-01, -4.3683e-01,
          9.6704e-01,  1.4703e-01, -1.6235e-01,  4.2479e-01, -4.2360e-01,
  

We use the model\'s prediction and the corresponding label to calculate
the error (`loss`). The next step is to backpropagate this error through
the network. Backward propagation is kicked off when we call
`.backward()` on the error tensor. Autograd then calculates and stores
the gradients for each model parameter in the parameter\'s `.grad`
attribute.


In [4]:
loss = (prediction - labels).sum()
print(loss)
loss.backward() # backward pass

tensor(-508.1966, grad_fn=<SumBackward0>)


Next, we load an optimizer, in this case SGD with a learning rate of
0.01 and
[momentum](https://medium.com/data-science/stochastic-gradient-descent-with-momentum-a84097641a5d)
of 0.9. We register all the parameters of the model in the optimizer.


In [5]:
optim = torch.optim.SGD(model.parameters(), lr=1e-2, momentum=0.9)

Finally, we call `.step()` to initiate gradient descent. The optimizer
adjusts each parameter by its gradient stored in `.grad`.


In [6]:
optim.step() #gradient descent

At this point, you have everything you need to train your neural
network. The below sections detail the workings of autograd - feel free
to skip them.


------------------------------------------------------------------------


Differentiation in Autograd
===========================

Let\'s take a look at how `autograd` collects gradients. We create two
tensors `a` and `b` with `requires_grad=True`. This signals to
`autograd` that every operation on them should be tracked.


In [7]:
import torch

a = torch.tensor([2., 3.], requires_grad=True)
b = torch.tensor([6., 4.], requires_grad=True)

We create another tensor `Q` from `a` and `b`.

$$Q = 3a^3 - b^2$$


In [8]:
Q = 3*a**3 - b**2
print(Q)

tensor([-12.,  65.], grad_fn=<SubBackward0>)


Let\'s assume `a` and `b` to be parameters of an NN, and `Q` to be the
error. In NN training, we want gradients of the error w.r.t. parameters,
i.e.

$$\frac{\partial Q}{\partial a} = 9a^2$$

$$\frac{\partial Q}{\partial b} = -2b$$

When we call `.backward()` on `Q`, autograd calculates these gradients
and stores them in the respective tensors\' `.grad` attribute.

We need to explicitly pass a `gradient` argument in `Q.backward()`
because it is a vector. `gradient` is a tensor of the same shape as `Q`,
and it represents the gradient of Q w.r.t. itself, i.e.

$$\frac{dQ}{dQ} = 1$$

Equivalently, we can also aggregate Q into a scalar and call backward
implicitly, like `Q.sum().backward()`.


In [9]:
Q.sum().backward()
print(Q)

tensor([-12.,  65.], grad_fn=<SubBackward0>)


Gradients are now deposited in `a.grad` and `b.grad`


In [10]:
# check if collected gradients are correct
print(9*a**2 == a.grad)
print(-2*b == b.grad)

tensor([True, True])
tensor([True, True])


In [17]:
# Paramètres de la couche 1
W1 = torch.tensor([[1., 2.], [3., 4.]], requires_grad=True)  # (2x2)
b1 = torch.tensor([0.5, 0.5], requires_grad=True)

# Paramètres de la couche 2
W2 = torch.tensor([[1., 0.], [0., 1.]], requires_grad=True)  # (2x2)
b2 = torch.tensor([0., 0.], requires_grad=True)

# Input
x = torch.tensor([1., 2.], requires_grad=True)

# Forward pass — 2 couches
h = torch.relu(x @ W1.T + b1) # couche 1 : shape (2,)
print("h")
print(h)
y = h @ W2.T + b2                # couche 2 : shape (2,)
print(y)
loss = y.sum()                   # scalaire pour pouvoir appeler backward()
loss.backward()

print("W2.grad :", W2.grad)
# [[h1, h2], [h1, h2]]  ← chaque ligne = h

print("W1.grad :", W1.grad)
# gradient remonté à travers ReLU et W2

print(b2.grad)
print(b1.grad)
print(x.grad)


h
tensor([ 5.5000, 11.5000], grad_fn=<ReluBackward0>)
tensor([ 5.5000, 11.5000], grad_fn=<AddBackward0>)
W2.grad : tensor([[ 5.5000, 11.5000],
        [ 5.5000, 11.5000]])
W1.grad : tensor([[1., 2.],
        [1., 2.]])
tensor([1., 1.])
tensor([1., 1.])
tensor([4., 6.])


Optional Reading - Vector Calculus using `autograd`
===================================================

Mathematically, if you have a vector valued function
$\vec{y}=f(\vec{x})$, then the gradient of $\vec{y}$ with respect to
$\vec{x}$ is a Jacobian matrix $J$:

$$\begin{aligned}
J
=
 \left(\begin{array}{cc}
 \frac{\partial \bf{y}}{\partial x_{1}} &
 ... &
 \frac{\partial \bf{y}}{\partial x_{n}}
 \end{array}\right)
=
\left(\begin{array}{ccc}
 \frac{\partial y_{1}}{\partial x_{1}} & \cdots & \frac{\partial y_{1}}{\partial x_{n}}\\
 \vdots & \ddots & \vdots\\
 \frac{\partial y_{m}}{\partial x_{1}} & \cdots & \frac{\partial y_{m}}{\partial x_{n}}
 \end{array}\right)
\end{aligned}$$

Generally speaking, `torch.autograd` is an engine for computing
vector-Jacobian product. That is, given any vector $\vec{v}$, compute
the product $J^{T}\cdot \vec{v}$

If $\vec{v}$ happens to be the gradient of a scalar function
$l=g\left(\vec{y}\right)$:

$$\vec{v}
 =
 \left(\begin{array}{ccc}\frac{\partial l}{\partial y_{1}} & \cdots & \frac{\partial l}{\partial y_{m}}\end{array}\right)^{T}$$

then by the chain rule, the vector-Jacobian product would be the
gradient of $l$ with respect to $\vec{x}$:

$$\begin{aligned}
J^{T}\cdot \vec{v} = \left(\begin{array}{ccc}
 \frac{\partial y_{1}}{\partial x_{1}} & \cdots & \frac{\partial y_{m}}{\partial x_{1}}\\
 \vdots & \ddots & \vdots\\
 \frac{\partial y_{1}}{\partial x_{n}} & \cdots & \frac{\partial y_{m}}{\partial x_{n}}
 \end{array}\right)\left(\begin{array}{c}
 \frac{\partial l}{\partial y_{1}}\\
 \vdots\\
 \frac{\partial l}{\partial y_{m}}
 \end{array}\right) = \left(\begin{array}{c}
 \frac{\partial l}{\partial x_{1}}\\
 \vdots\\
 \frac{\partial l}{\partial x_{n}}
 \end{array}\right)
\end{aligned}$$

This characteristic of vector-Jacobian product is what we use in the
above example; `external_grad` represents $\vec{v}$.


Computational Graph
===================

Conceptually, autograd keeps a record of data (tensors) & all executed
operations (along with the resulting new tensors) in a directed acyclic
graph (DAG) consisting of
[Function](https://pytorch.org/docs/stable/autograd.html#torch.autograd.Function)
objects. In this DAG, leaves are the input tensors, roots are the output
tensors. By tracing this graph from roots to leaves, you can
automatically compute the gradients using the chain rule.

In a forward pass, autograd does two things simultaneously:

-   run the requested operation to compute a resulting tensor, and
-   maintain the operation's *gradient function* in the DAG.

The backward pass kicks off when `.backward()` is called on the DAG
root. `autograd` then:

-   computes the gradients from each `.grad_fn`,
-   accumulates them in the respective tensor's `.grad` attribute, and
-   using the chain rule, propagates all the way to the leaf tensors.

Below is a visual representation of the DAG in our example. In the
graph, the arrows are in the direction of the forward pass. The nodes
represent the backward functions of each operation in the forward pass.
The leaf nodes in blue represent our leaf tensors `a` and `b`.

![](https://pytorch.org/tutorials/_static/img/dag_autograd.png)

<div style="background-color: #54c7ec; color: #fff; font-weight: 700; padding-left: 10px; padding-top: 5px; padding-bottom: 5px"><strong>NOTE:</strong></div>

<div style="background-color: #f3f4f7; padding-left: 10px; padding-top: 10px; padding-bottom: 10px; padding-right: 10px">

<p>An important thing to note is that the graph is recreated from scratch; after each<code>.backward()</code> call, autograd starts populating a new graph. This isexactly what allows you to use control flow statements in your model;you can change the shape, size and operations at every iteration ifneeded.</p>

</div>

Exclusion from the DAG
----------------------

`torch.autograd` tracks operations on all tensors which have their
`requires_grad` flag set to `True`. For tensors that don't require
gradients, setting this attribute to `False` excludes it from the
gradient computation DAG.

The output tensor of an operation will require gradients even if only a
single input tensor has `requires_grad=True`.


In [12]:
x = torch.rand(5, 5)
y = torch.rand(5, 5)
z = torch.rand((5, 5), requires_grad=True)

a = x + y
print(f"Does `a` require gradients?: {a.requires_grad}")
b = x + z
print(f"Does `b` require gradients?: {b.requires_grad}")

Does `a` require gradients?: False
Does `b` require gradients?: True


In a NN, parameters that don\'t compute gradients are usually called
**frozen parameters**. It is useful to \"freeze\" part of your model if
you know in advance that you won\'t need the gradients of those
parameters (this offers some performance benefits by reducing autograd
computations).

In finetuning, we freeze most of the model and typically only modify the
classifier layers to make predictions on new labels. Let\'s walk through
a small example to demonstrate this. As before, we load a pretrained
resnet18 model, and freeze all the parameters.


In [19]:
from torch import nn, optim

model = resnet18(weights=ResNet18_Weights.DEFAULT)
print(model.parameters)

# Freeze all the parameters in the network
for param in model.parameters():
    param.requires_grad = False

<bound method Module.parameters of ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)


Let\'s say we want to finetune the model on a new dataset with 10
labels. In resnet, the classifier is the last linear layer `model.fc`.
We can simply replace it with a new linear layer (unfrozen by default)
that acts as our classifier.


In [20]:
model.fc = nn.Linear(512, 10)

Now all parameters in the model, except the parameters of `model.fc`,
are frozen. The only parameters that compute gradients are the weights
and bias of `model.fc`.


In [21]:
# Optimize only the classifier
optimizer = optim.SGD(model.parameters(), lr=1e-2, momentum=0.9)

Notice although we register all the parameters in the optimizer, the
only parameters that are computing gradients (and hence updated in
gradient descent) are the weights and bias of the classifier.

The same exclusionary functionality is available as a context manager in
[torch.no\_grad()](https://pytorch.org/docs/stable/generated/torch.no_grad.html)


------------------------------------------------------------------------


Further readings:
=================

-   [In-place operations & Multithreaded
    Autograd](https://pytorch.org/docs/stable/notes/autograd.html)
-   [Example implementation of reverse-mode
    autodiff](https://colab.research.google.com/drive/1VpeE6UvEPRz9HmsHh1KS0XxXjYu533EC)
-   [Video: PyTorch Autograd Explained - In-depth
    Tutorial](https://www.youtube.com/watch?v=MswxJw-8PvE)
